# Evaluate a trained DPO checkpoint

Loads a checkpoint saved by `train.py`, computes pairwise preference accuracy on held-out UltraFeedback pairs, and optionally samples a greedy completion for inspection.

**Prerequisite**: a completed training run. The default `CKPT` path below matches the smoke config:

```
python train.py --config configs/gpt2_cpu_smoke.yaml
```

~10-15 min on a laptop CPU, a few seconds on a GPU. The cell below will tell you if the checkpoint isn't there yet.

In [ ]:
# Fresh-kernel setup (Colab or a Jupyter kernel without the repo's deps).
# Safe to re-run: it checks before installing or cloning.
import importlib.util, os
if importlib.util.find_spec('jax') is None:
    !pip install -q -U jax flax optax numpy "transformers>=4.44,<5" datasets sentencepiece pyyaml
if not os.path.isfile('losses.py'):
    !git clone -q https://github.com/jman4162/jax-dpo-min.git
    %cd jax-dpo-min

In [ ]:
CKPT = 'outputs/gpt2_cpu_smoke/step_000200'

import pickle
from pathlib import Path

import jax
import jax.numpy as jnp

ckpt_file = Path(CKPT) / 'state.pkl'
if not ckpt_file.exists():
    print(f'No checkpoint at {ckpt_file}.')
    print('Train one first:')
    print('    python train.py --config configs/gpt2_cpu_smoke.yaml')
    print('Then re-run this cell.')
else:
    from datasets import load_dataset

    from eval import pairwise_accuracy
    from model import load_model_and_tokenizer

    with open(ckpt_file, 'rb') as f:
        state = pickle.load(f)
    cfg = state['config']
    lora_params = jax.tree_util.tree_map(jnp.asarray, state['lora_params'])

    model, base_params, tokenizer = load_model_and_tokenizer(cfg['model'], dtype=cfg.get('dtype', 'float32'))
    ds = load_dataset(cfg['dataset'], split='train[-64:]')
    acc = pairwise_accuracy(model, base_params, lora_params, tokenizer, ds, cfg)
    print(f'pairwise_accuracy on {len(ds)} held-out pairs: {acc:.3f}')

## Optional: sample a completion

Greedy decoding from the DPO-finetuned policy. Slow on CPU — skip if that's your runtime.

In [ ]:
if ckpt_file.exists():
    from eval import greedy_sample
    from lora import merge

    merged = merge(base_params, lora_params, alpha=cfg['lora_alpha'], rank=cfg['lora_rank'])
    prompt = 'Explain in one sentence why the DPO loss uses a sigmoid:'
    print(greedy_sample(model, merged, tokenizer, prompt, max_new_tokens=48))